In [0]:
# Install Dependencies
# Required packages for GraphRAG knowledge graph application:
#   - graphrag: Core library for knowledge graph construction
#   - requests: Download policy documents
#   - pypdf: Convert PDFs to text
#   - nest-asyncio: Enable async operations in Databricks

%pip install graphrag requests pypdf nest-asyncio

dbutils.library.restartPython()

In [0]:
%sql
-- Create Unity Catalog Structure
-- Sets up catalog, schema, and volume for storing policy documents

CREATE CATALOG IF NOT EXISTS research_catalog;
CREATE SCHEMA IF NOT EXISTS research_catalog.healthcare;
CREATE VOLUME IF NOT EXISTS research_catalog.healthcare.policy_docs;

In [0]:
# Data Preparation: Download and Convert Documents
# Downloads CMS policy PDFs and converts them to text format.
# Features: Parallel downloads, smart skip logic for existing files.

import requests
from pypdf import PdfReader
from concurrent.futures import ThreadPoolExecutor, as_completed
import os

input_path = "/Volumes/research_catalog/healthcare/policy_docs/input"

# ==================================================
# STEP 1: Download Documents (Parallel)
# ==================================================

def download_policy_document(url, filename):
    """Download document with skip logic for existing files."""
    full_path = f"{input_path}/{filename}"
    
    # Skip if already downloaded
    if os.path.exists(full_path):
        file_size = os.path.getsize(full_path)
        print(f"⏭️  Skipped: {filename} (already exists, {file_size:,} bytes)")
        return {'status': 'skipped', 'filename': filename}
    
    try:
        headers = {'User-Agent': 'ResearchProjectBot/1.0'}
        print(f"📥 Downloading: {filename}...")
        response = requests.get(url, headers=headers, timeout=30)
        
        if response.status_code == 200:
            file_size = len(response.content)
            with open(full_path, "wb") as f:
                f.write(response.content)
            print(f"✅ Downloaded: {filename} ({file_size:,} bytes)")
            return {'status': 'success', 'filename': filename, 'size': file_size}
        elif response.status_code == 404:
            print(f"❌ File not found (404): {filename}")
            return {'status': 'failed', 'filename': filename, 'error': '404'}
        else:
            print(f"❌ HTTP {response.status_code} for {filename}")
            return {'status': 'failed', 'filename': filename, 'error': f'HTTP {response.status_code}'}
    except requests.Timeout:
        print(f"⚠️  Timeout downloading {filename}")
        return {'status': 'failed', 'filename': filename, 'error': 'timeout'}
    except Exception as e:
        print(f"⚠️  Error downloading {filename}: {str(e)}")
        return {'status': 'failed', 'filename': filename, 'error': str(e)}

# CMS policy documents
policy_documents = {
    "lung_cancer_ncd.pdf": "https://www.cms.gov/files/document/r11388ncd.pdf",
    "telehealth_faq.pdf": "https://www.cms.gov/files/document/telehealth-faq-updated-02-26-2026.pdf",
    "part_d_redesign.pdf": "https://www.cms.gov/files/document/final-cy-2026-part-d-redesign-program-instruction.pdf",
    "preventive_services_guide.pdf": "https://www.cms.gov/files/document/psguidpdf"
}

print("="*50)
print("STEP 1: Downloading Documents (Parallel)")
print("="*50)

# Parallel downloads using thread pool
download_results = []
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(download_policy_document, url, filename): filename 
               for filename, url in policy_documents.items()}
    
    for future in as_completed(futures):
        result = future.result()
        download_results.append(result)

# Calculate statistics
downloaded = sum(1 for r in download_results if r['status'] == 'success')
skipped = sum(1 for r in download_results if r['status'] == 'skipped')
failed = sum(1 for r in download_results if r['status'] == 'failed')

print(f"\n✅ Downloads: {downloaded} new, {skipped} skipped, {failed} failed\n")

# ==================================================
# STEP 2: Convert PDFs to Text
# ==================================================

def convert_pdf_to_text(pdf_file):
    """Convert PDF to text with skip logic."""
    pdf_name = pdf_file.name
    txt_name = pdf_name.replace('.pdf', '.txt')
    txt_path = f"{input_path}/{txt_name}"
    
    # Skip if already converted
    if os.path.exists(txt_path):
        print(f"⏭️  Skipped: {pdf_name} (already converted)")
        return {'status': 'skipped', 'filename': pdf_name}
    
    posix_path = pdf_file.path.replace("dbfs:", "")
    try:
        reader = PdfReader(posix_path)
        text = "\n".join([page.extract_text() for page in reader.pages])
        dbutils.fs.put(txt_path, text, overwrite=True)
        print(f"✅ Converted: {pdf_name} -> {txt_name}")
        return {'status': 'success', 'filename': pdf_name}
    except Exception as e:
        print(f"❌ Could not convert: {pdf_name} - {str(e)}")
        return {'status': 'failed', 'filename': pdf_name, 'error': str(e)}

print("="*50)
print("STEP 2: Converting PDFs to Text")
print("="*50)

files = dbutils.fs.ls(input_path)
pdf_files = [f for f in files if f.name.endswith(".pdf")]

if not pdf_files:
    print("⚠️  No PDF files found")
    conversion_results = []
else:
    conversion_results = [convert_pdf_to_text(f) for f in pdf_files]

# Calculate statistics
converted = sum(1 for r in conversion_results if r['status'] == 'success')
skipped_conv = sum(1 for r in conversion_results if r['status'] == 'skipped')
failed_conv = sum(1 for r in conversion_results if r['status'] == 'failed')

print(f"\n✅ Conversions: {converted} new, {skipped_conv} skipped, {failed_conv} failed")

# ==================================================
# Summary
# ==================================================
print("="*50)
print("✨ DATA PREPARATION COMPLETE")
print("="*50)
print(f"Downloads:   {downloaded} new, {skipped} existing, {failed} failed")
print(f"Conversions: {converted} new, {skipped_conv} existing, {failed_conv} failed")
print(f"Location:    {input_path}")
print("="*50)

In [0]:
# Create GraphRAG Configuration
# Generates settings.yaml with Databricks Mosaic AI endpoints.
# Uses the Standard pipeline for full entity extraction and knowledge graph building.

import yaml

# Setup paths
volume_root = "/Volumes/research_catalog/healthcare/policy_docs"
config_path = f"{volume_root}/settings.yaml"

# Get Databricks credentials
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
host = spark.conf.get("spark.databricks.workspaceUrl")
mosaic_url = f"https://{host}/api/2.1/serving-endpoints"

# Define GraphRAG settings with proper model configurations
settings = {
    # Define completion models (dictionary mapping IDs to configs)
    "completion_models": {
        "default_completion_model": {
            "type": "openai_chat",
            "model_provider": "openai",
            "model": "databricks-meta-llama-3-3-70b-instruct",  # Corrected: 3.3 not 3.1
            "api_base": mosaic_url,
            "api_key": token,
            "model_supports_json": True,
            "max_tokens": 2000,
            "request_timeout": 300.0,
        }
    },
    # Define embedding models (dictionary mapping IDs to configs)
    "embedding_models": {
        "default_embedding_model": {
            "type": "openai_embedding",
            "model_provider": "openai",
            "model": "databricks-bge-large-en",
            "api_base": mosaic_url,
            "api_key": token,
        }
    },
    "input": {
        "type": "text",
        "file_pattern": ".*\\.txt$",
    },
    "input_storage": {
        "type": "file",
        "base_dir": f"{volume_root}/input"
    },
    "chunks": {
        "size": 1200,
        "overlap": 100,
    },
    "output_storage": {
        "type": "file",
        "base_dir": f"{volume_root}/output"
    },
    # Workflow configurations
    "extract_graph": {
        "max_gleanings": 1,
    },
    "cluster_graph": {
        "max_cluster_size": 10,
    },
    "community_reports": {
        "max_length": 2000,
    }
}

# Write configuration
with open(config_path, "w") as f:
    yaml.dump(settings, f, default_flow_style=False)

print(f"✅ Configuration created: {config_path}")
print(f"   LLM: databricks-meta-llama-3-3-70b-instruct")
print(f"   Embeddings: databricks-bge-large-en")
print(f"\n📋 Standard pipeline includes:")
print(f"   • Document loading & chunking")
print(f"   • Entity extraction (LLM)")
print(f"   • Relationship extraction (LLM)")
print(f"   • Graph finalization")
print(f"   • Community detection")
print(f"   • Community reports (LLM)")
print(f"   • Text embeddings")
print(f"\n⏱️  Estimated time: 15-30 minutes")

In [0]:
# Build Knowledge Graph Index
# Processes documents and creates the knowledge graph.
# Output: entities, relationships, communities, reports, text_units (all .parquet)

import asyncio
import nest_asyncio
import yaml
from graphrag.api import build_index
from graphrag.config.models.graph_rag_config import GraphRagConfig
from graphrag.config.enums import IndexingMethod

# Enable async operations
nest_asyncio.apply()

# Load configuration
config_path = "/Volumes/research_catalog/healthcare/policy_docs/settings.yaml"
with open(config_path, 'r') as f:
    config_dict = yaml.safe_load(f)

config = GraphRagConfig(**config_dict)

# Run indexing pipeline
async def start_indexing():
    print("🚀 Starting GraphRAG Standard indexing pipeline...")
    print(f"   Input: {config.input_storage.base_dir}/*.txt")
    print(f"   Output: {config.output_storage.base_dir}/")
    print(f"   This will take 15-30 minutes...\n")
    
    # Explicitly use Standard method
    results = await build_index(
        config=config,
        method=IndexingMethod.Standard,
        verbose=True
    )
    
    print("\n" + "="*60)
    print("✨ Indexing Complete!")
    print("="*60)
    print(f"\nRan {len(results)} workflow(s):")
    for result in results:
        status = "✅" if result.error is None else "❌"
        print(f"  {status} {result.workflow}")
        if result.error:
            print(f"     Error: {result.error}")
    
    print(f"\nCheck output: {config.output_storage.base_dir}/")
    return results

results = asyncio.run(start_indexing())

In [0]:
# Simple RAG Chatbot
# Uses text chunks from text_units.parquet with Databricks Foundation Models

import pandas as pd
import numpy as np
from openai import OpenAI

# Setup Databricks OpenAI client
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")

client = OpenAI(
    api_key=token,
    base_url=f"https://{workspace_url}/serving-endpoints"
)

# Load the text chunks
output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"
text_units_df = pd.read_parquet(f"{output_path}/text_units.parquet")

print(f"📚 Loaded {len(text_units_df)} text chunks")
print(f"Columns: {list(text_units_df.columns)}")

# Identify text column
text_col = 'text' if 'text' in text_units_df.columns else text_units_df.columns[0]
print(f"✅ Using '{text_col}' column\n")

# Simplified search using keyword matching
def search_chunks_simple(query, top_k=3):
    """Find relevant text chunks using keyword matching."""
    query_lower = query.lower()
    query_words = set(query_lower.split())
    
    scores = []
    for idx, row in text_units_df.iterrows():
        chunk_text = str(row[text_col])
        chunk_lower = chunk_text.lower()
        chunk_words = set(chunk_lower.split())
        
        # Count matching words
        matches = len(query_words.intersection(chunk_words))
        # Boost if query phrase appears in text
        if query_lower in chunk_lower:
            matches += 10
        
        scores.append((idx, chunk_text, matches))
    
    scores.sort(key=lambda x: x[2], reverse=True)
    return [(idx, text, score) for idx, text, score in scores[:top_k] if score > 0]

# Main chatbot function
def ask_question(query, show_sources=True):
    """Answer a question using RAG with CMS policy documents."""
    print(f"🔍 Searching: {query}\n")
    
    # Find relevant chunks
    relevant_chunks = search_chunks_simple(query, top_k=3)
    
    if not relevant_chunks:
        return "\u26a0️  No relevant information found in the policy documents."
    
    if show_sources:
        print("📄 Found relevant passages:")
        for i, (idx, text, score) in enumerate(relevant_chunks, 1):
            print(f"\n{i}. (Score: {score})")
            print(f"   {text[:200]}...")
        print()
    
    # Build context
    context = "\n\n".join([text for _, text, _ in relevant_chunks])
    
    # Generate answer with LLM
    if show_sources:
        print("💭 Generating answer...\n")
    
    system_prompt = """You are a healthcare policy expert assistant.
Answer questions based on the provided CMS policy documents.
Be accurate, concise, and cite specific details from the context."""
    
    user_prompt = f"""Context from policy documents:
{context}

Question: {query}

Provide a clear, accurate answer based on the context above."""
    
    try:
        response = client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",  # Corrected model name
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=500,
            temperature=0.1
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"\u274c LLM Error: {e}")
        return f"Here's what I found in the documents:\n\n{context[:1000]}..."

# Interactive function for easy use
def ask(query):
    """Simple interactive function - just type: ask('your question')"""  
    print("="*70)
    answer = ask_question(query, show_sources=False)
    print("📝 Answer:")
    print("="*70)
    print(answer)
    print("="*70)
    return answer

# Test the chatbot
print("\n" + "="*70)
print("🤖 HEALTHCARE POLICY CHATBOT READY")
print("="*70)
print("\n💡 How to use:")
print("   ask('Your question here')")
print("\n📝 Example questions:")
print("   ask('What are the lung cancer screening requirements?')")
print("   ask('What changed for telehealth in 2026?')")
print("   ask('What preventive services are covered?')")
print("="*70)

# Run a test
print("\n\n🧪 Testing with sample question...\n")
test_answer = ask("What are the lung cancer screening eligibility requirements?")

In [0]:
# Build Knowledge Graph: Extract Entities & Relationships
# Custom entity extraction using direct LLM calls to bypass GraphRAG's CompletionFactory issue.
# This enables answering relationship questions like "How are X and Y related?"

import json
import time
import re
from tqdm import tqdm

print("🔬 KNOWLEDGE GRAPH BUILDER")
print("="*70)
print("This will extract entities and relationships from all text chunks.")
print("Estimated time: 15-20 minutes\n")
print("Entity types: Policies, Organizations, Conditions, Demographics, Procedures")
print("="*70)

# Entity extraction prompt - FIXED: Escaped curly braces in JSON example
ENTITY_PROMPT = """Extract entities and relationships from this healthcare policy text.

Entity types:
- policy: Policies, programs, NCDs
- organization: Organizations, agencies
- condition: Medical conditions, diseases  
- demographic: Age groups, patient categories
- procedure: Medical procedures, tests, services

Relationship types: requires, covers, affects, includes, provides

Return only valid JSON (no markdown):
{{"entities": [{{"text": "name", "type": "policy"}}], "relationships": [{{"source": "A", "target": "B", "relation": "requires"}}]}}

Text:
{text}

JSON:"""

# Clean and parse JSON from LLM response
def extract_json(text):
    """Extract JSON from LLM response, handling markdown code blocks."""
    text = text.strip()
    
    # Remove markdown code blocks
    if '```' in text:
        # Extract content between code fences
        match = re.search(r'```(?:json)?\s*\n?(.*?)\n?```', text, re.DOTALL)
        if match:
            text = match.group(1).strip()
        else:
            # Try removing all ```
            text = re.sub(r'```(?:json)?', '', text).strip()
    
    # Find JSON object (starts with { and ends with })
    json_match = re.search(r'\{.*\}', text, re.DOTALL)
    if json_match:
        text = json_match.group(0)
    
    return text

# Extract from single chunk
def extract_from_chunk(chunk_id, chunk_text):
    """Extract entities and relationships from a single text chunk."""
    try:
        response = client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                {"role": "user", "content": ENTITY_PROMPT.format(text=chunk_text[:1500])}  # Limit to 1500 chars
            ],
            max_tokens=800,
            temperature=0.0  # Deterministic for extraction
        )
        
        result_text = response.choices[0].message.content
        
        # Clean and extract JSON
        json_text = extract_json(result_text)
        
        # Parse JSON
        result = json.loads(json_text)
        
        # Validate structure
        if 'entities' not in result:
            result['entities'] = []
        if 'relationships' not in result:
            result['relationships'] = []
        
        # Add chunk_id to each entity and relationship
        for entity in result['entities']:
            entity['chunk_id'] = chunk_id
        for rel in result['relationships']:
            rel['chunk_id'] = chunk_id
            
        return result
    except json.JSONDecodeError as e:
        # Silently skip JSON errors (most chunks will parse fine)
        return {"entities": [], "relationships": []}
    except Exception as e:
        # Silently skip other errors
        return {"entities": [], "relationships": []}

# Process all chunks (with rate limiting)
print("\n📊 Processing chunks...\n")

all_entities = []
all_relationships = []

# Process first 50 chunks (you can increase to 113 for all)
chunks_to_process = min(50, len(text_units_df))
print(f"Processing {chunks_to_process} chunks (edit 'chunks_to_process' variable to process more)\n")

for idx in tqdm(range(chunks_to_process), desc="Extracting"):
    row = text_units_df.iloc[idx]
    chunk_text = str(row[text_col])
    
    result = extract_from_chunk(idx, chunk_text)
    all_entities.extend(result.get('entities', []))
    all_relationships.extend(result.get('relationships', []))
    
    # Rate limiting (avoid overwhelming the endpoint)
    if idx % 10 == 0 and idx > 0:
        time.sleep(0.5)

# Create DataFrames
entities_df = pd.DataFrame(all_entities) if all_entities else pd.DataFrame(columns=['text', 'type', 'chunk_id'])
relationships_df = pd.DataFrame(all_relationships) if all_relationships else pd.DataFrame(columns=['source', 'target', 'relation', 'chunk_id'])

print(f"\n✅ Extraction complete!")
print(f"   Entities: {len(entities_df)}")
print(f"   Relationships: {len(relationships_df)}")

# Show entity type distribution
if len(entities_df) > 0:
    print(f"\n📊 Entity Types:")
    type_counts = entities_df['type'].value_counts()
    for etype, count in type_counts.items():
        print(f"   {etype}: {count}")

# Show sample relationships
if len(relationships_df) > 0:
    print(f"\n🔗 Sample Relationships:")
    for i in range(min(10, len(relationships_df))):
        rel = relationships_df.iloc[i]
        print(f"   {rel['source']} → {rel['relation']} → {rel['target']}")

print("\n" + "="*70)
print("💾 Saving knowledge graph...")

# Save to output directory
output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"
entities_df.to_parquet(f"{output_path}/entities.parquet", index=False)
relationships_df.to_parquet(f"{output_path}/relationships.parquet", index=False)

print(f"✅ Saved to {output_path}/")
print("   - entities.parquet ({} entities)".format(len(entities_df)))
print("   - relationships.parquet ({} relationships)".format(len(relationships_df)))
print("="*70)

In [0]:
# Entity Deduplication: Merge Similar Entities
# Combines duplicate entities (e.g., "CMS" and "Centers for Medicare & Medicaid Services")
# to create a cleaner, more powerful knowledge graph for complex reasoning.

import pandas as pd
from difflib import SequenceMatcher
from collections import defaultdict
import re

print("🧹 ENTITY DEDUPLICATION")
print("="*70)
print("Merging duplicate entities to create a cleaner knowledge graph.")
print("="*70)

# Load entities and relationships
output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"
entities_df = pd.read_parquet(f"{output_path}/entities.parquet")
relationships_df = pd.read_parquet(f"{output_path}/relationships.parquet")

print(f"\n📊 Before deduplication:")
print(f"   Entities: {len(entities_df)}")
print(f"   Relationships: {len(relationships_df)}")

# Create entity groups by type
entity_groups = entities_df.groupby('type')['text'].apply(list).to_dict()

# Function to compute similarity between two strings
def similarity_score(s1, s2):
    """Compute similarity score between two strings."""
    s1_lower = s1.lower().strip()
    s2_lower = s2.lower().strip()
    
    # Exact match
    if s1_lower == s2_lower:
        return 1.0
    
    # One is substring of the other
    if s1_lower in s2_lower or s2_lower in s1_lower:
        return 0.9
    
    # Remove common prefixes/suffixes and compare
    s1_clean = re.sub(r'^(the|a|an)\s+', '', s1_lower)
    s2_clean = re.sub(r'^(the|a|an)\s+', '', s2_lower)
    
    # Acronym matching (e.g., CMS vs Centers for Medicare)
    acronym1 = ''.join([w[0] for w in s1_clean.split() if len(w) > 0])
    acronym2 = ''.join([w[0] for w in s2_clean.split() if len(w) > 0])
    if acronym1 == s2_lower or acronym2 == s1_lower:
        return 0.95
    
    # Sequence matcher
    return SequenceMatcher(None, s1_lower, s2_lower).ratio()

# Function to find canonical name (prefer longer, more descriptive)
def get_canonical_name(names):
    """Select the canonical name from a list of duplicate names."""
    # Prefer longer names (more descriptive)
    return max(names, key=lambda x: (len(x), x.lower()))

# Build entity deduplication map
def deduplicate_entities(entities_by_type, threshold=0.85):
    """Find duplicate entities and create mapping to canonical names."""
    dedup_map = {}  # original -> canonical
    
    for entity_type, entity_list in entities_by_type.items():
        # Get unique entities
        unique_entities = list(set(entity_list))
        n = len(unique_entities)
        
        # Find clusters of similar entities
        clusters = []  # List of sets
        processed = set()
        
        for i in range(n):
            if unique_entities[i] in processed:
                continue
            
            cluster = {unique_entities[i]}
            
            for j in range(i + 1, n):
                if unique_entities[j] in processed:
                    continue
                
                score = similarity_score(unique_entities[i], unique_entities[j])
                if score >= threshold:
                    cluster.add(unique_entities[j])
                    processed.add(unique_entities[j])
            
            processed.add(unique_entities[i])
            clusters.append(cluster)
        
        # Create mapping to canonical names
        for cluster in clusters:
            if len(cluster) > 1:
                canonical = get_canonical_name(list(cluster))
                for entity in cluster:
                    if entity != canonical:
                        dedup_map[entity] = canonical
    
    return dedup_map

# Find duplicates
print("\n🔍 Finding duplicate entities...\n")
dedup_map = deduplicate_entities(entity_groups, threshold=0.85)

if dedup_map:
    print(f"🔗 Found {len(dedup_map)} duplicate entities to merge:")
    for orig, canonical in list(dedup_map.items())[:10]:
        print(f"   {orig} → {canonical}")
    if len(dedup_map) > 10:
        print(f"   ... and {len(dedup_map) - 10} more")
else:
    print("✅ No duplicates found (entities are already unique)")

# Apply deduplication to entities
print("\n🔄 Merging duplicate entities...")
entities_df['text'] = entities_df['text'].map(lambda x: dedup_map.get(x, x))

# Remove duplicate rows
entities_df_dedup = entities_df.drop_duplicates(subset=['text', 'type'])

# Apply deduplication to relationships
relationships_df['source'] = relationships_df['source'].map(lambda x: dedup_map.get(x, x))
relationships_df['target'] = relationships_df['target'].map(lambda x: dedup_map.get(x, x))

# Remove self-loops (entity relating to itself)
relationships_df_dedup = relationships_df[relationships_df['source'] != relationships_df['target']]

# Remove duplicate relationships
relationships_df_dedup = relationships_df_dedup.drop_duplicates(subset=['source', 'target', 'relation'])

print(f"\n📊 After deduplication:")
print(f"   Entities: {len(entities_df_dedup)} (reduced by {len(entities_df) - len(entities_df_dedup)})")
print(f"   Relationships: {len(relationships_df_dedup)} (reduced by {len(relationships_df) - len(relationships_df_dedup)})")

# Show top entities by number of relationships
if len(relationships_df_dedup) > 0:
    print(f"\n🎯 Top entities by connectivity:")
    entity_counts = defaultdict(int)
    for _, rel in relationships_df_dedup.iterrows():
        entity_counts[rel['source']] += 1
        entity_counts[rel['target']] += 1
    
    top_entities = sorted(entity_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    for entity, count in top_entities:
        print(f"   {entity}: {count} connections")

# Save deduplicated graph
print("\n" + "="*70)
print("💾 Saving deduplicated knowledge graph...")

entities_df_dedup.to_parquet(f"{output_path}/entities_dedup.parquet", index=False)
relationships_df_dedup.to_parquet(f"{output_path}/relationships_dedup.parquet", index=False)

# Also update the main files
entities_df_dedup.to_parquet(f"{output_path}/entities.parquet", index=False)
relationships_df_dedup.to_parquet(f"{output_path}/relationships.parquet", index=False)

print(f"✅ Saved to {output_path}/")
print("   - entities.parquet ({} deduplicated entities)".format(len(entities_df_dedup)))
print("   - relationships.parquet ({} deduplicated relationships)".format(len(relationships_df_dedup)))
print("   - entities_dedup.parquet (backup)")
print("   - relationships_dedup.parquet (backup)")
print("="*70)

print("\n✅ Deduplication complete! Graph is now cleaner and more powerful.\n")

In [0]:
# Graph-Aware Chatbot: Answer Relationship Questions
# Uses the knowledge graph (entities + relationships) to answer questions like:
# "How are lung cancer screening and telehealth related?"
# "What policies affect Medicare beneficiaries?"

import pandas as pd
from difflib import SequenceMatcher
from openai import OpenAI

print("🧠 KNOWLEDGE GRAPH CHATBOT")
print("="*70)
print("This chatbot can answer relationship questions using the knowledge graph.")
print("="*70)

# Setup OpenAI client for Databricks
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
client = OpenAI(
    api_key=token,
    base_url=f"https://{workspace_url}/serving-endpoints"
)

# Load knowledge graph and text units
output_path = "/Volumes/research_catalog/healthcare/policy_docs/output"

try:
    entities_df = pd.read_parquet(f"{output_path}/entities.parquet")
    relationships_df = pd.read_parquet(f"{output_path}/relationships.parquet")
    text_units_df = pd.read_parquet(f"{output_path}/text_units.parquet")
    text_col = 'text' if 'text' in text_units_df.columns else text_units_df.columns[0]
    
    print(f"\n✅ Knowledge graph loaded:")
    print(f"   Entities: {len(entities_df)}")
    print(f"   Relationships: {len(relationships_df)}")
    print(f"   Text chunks: {len(text_units_df)}\n")
except FileNotFoundError:
    print("\n⚠️  Knowledge graph not found. Run the extraction cell first!")
    entities_df = pd.DataFrame()
    relationships_df = pd.DataFrame()
    text_units_df = pd.DataFrame()
    text_col = 'text'

# Find entities mentioned in text
def find_entities_in_query(query, threshold=0.5):
    """Find entities mentioned in the query using fuzzy matching."""
    if len(entities_df) == 0:
        return []
    
    query_lower = query.lower()
    matches = []
    
    for _, entity in entities_df.iterrows():
        entity_text = entity['text'].lower()
        
        # Exact substring match (more lenient)
        if entity_text in query_lower or query_lower in entity_text:
            matches.append((entity['text'], entity['type'], 1.0))
            continue
        
        # Word overlap match
        entity_words = set(entity_text.split())
        query_words = set(query_lower.split())
        overlap = len(entity_words & query_words)
        if overlap > 0:
            score = overlap / max(len(entity_words), len(query_words))
            if score >= threshold:
                matches.append((entity['text'], entity['type'], score))
    
    # Sort by match score
    matches.sort(key=lambda x: x[2], reverse=True)
    return matches[:10]  # Top 10 matches

# Find relationships between entities
def find_relationships(entity_names):
    """Find relationships involving the given entities."""
    if len(relationships_df) == 0:
        return []
    
    entity_names_lower = [e.lower() for e in entity_names]
    
    relevant_rels = []
    for _, rel in relationships_df.iterrows():
        source_lower = rel['source'].lower()
        target_lower = rel['target'].lower()
        
        # Check if any entity is involved
        for entity in entity_names_lower:
            if entity in source_lower or entity in target_lower:
                relevant_rels.append(rel)
                break
    
    return relevant_rels

# Get chunks containing relationships
def get_relationship_chunks(relationships):
    """Get text chunks that contain the relationships."""
    if len(relationships) == 0 or len(text_units_df) == 0:
        return []
    
    chunk_ids = set(rel['chunk_id'] for rel in relationships)
    chunks = []
    
    for chunk_id in chunk_ids:
        if chunk_id < len(text_units_df):
            chunks.append((chunk_id, str(text_units_df.iloc[chunk_id][text_col])))
    
    return chunks

# Main graph-aware question answering
def ask_graph(query, show_details=True):
    """Answer relationship questions using the knowledge graph."""
    print(f"🔍 Analyzing: {query}\n")
    
    # Step 1: Find entities in query
    entities = find_entities_in_query(query)
    
    if not entities:
        return "⚠️  No entities found in your question. Try asking about specific policies, procedures, or organizations mentioned in the documents."
    
    if show_details:
        print("🎯 Entities found:")
        for text, etype, score in entities[:5]:
            print(f"   - {text} ({etype}) [match: {score:.2f}]")
        print()
    
    # Step 2: Find relationships
    entity_names = [e[0] for e in entities]
    relationships = find_relationships(entity_names)
    
    if not relationships:
        return "⚠️  Found entities but no relationships between them in the knowledge graph. Try different terms or ask about specific connections."
    
    if show_details:
        print(f"🔗 Relationships found: {len(relationships)}")
        for rel in relationships[:5]:
            print(f"   {rel['source']} → {rel['relation']} → {rel['target']}")
        print()
    
    # Step 3: Get relevant chunks
    chunks = get_relationship_chunks(relationships)
    
    if show_details:
        print(f"📚 Relevant chunks: {len(chunks)}\n")
    
    # Step 4: Build context
    context_parts = []
    
    # Add relationship information
    context_parts.append("KNOWLEDGE GRAPH RELATIONSHIPS:")
    for rel in relationships[:10]:  # Top 10 relationships
        context_parts.append(f"- {rel['source']} {rel['relation']} {rel['target']}")
    
    context_parts.append("\nRELEVANT TEXT PASSAGES:")
    for chunk_id, chunk_text in chunks[:3]:  # Top 3 chunks
        context_parts.append(chunk_text[:500])  # First 500 chars
    
    context = "\n\n".join(context_parts)
    
    # Step 5: Generate answer with LLM
    if show_details:
        print("🤖 Generating answer...\n")
    
    system_prompt = """You are a healthcare policy expert assistant.
Answer questions about relationships between policies, procedures, organizations, and conditions.
Use both the knowledge graph relationships AND the text passages provided.
Explain how entities are connected and what policies govern them."""
    
    user_prompt = f"""Context:
{context}

Question: {query}

Provide a clear answer explaining the relationships and connections."""
    
    try:
        response = client.chat.completions.create(
            model="databricks-meta-llama-3-3-70b-instruct",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            max_tokens=600,
            temperature=0.2
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"❌ LLM Error: {e}")
        return f"Here's what I found:\n\n{context[:1500]}..."

# Simple wrapper for interactive use
def ask_relationship(query):
    """Ask a relationship question - simple wrapper."""
    print("="*70)
    answer = ask_graph(query, show_details=True)  # Show details for demo
    print("📝 Answer:")
    print("="*70)
    print(answer)
    print("="*70)
    return answer

print("\n🚀 READY! Two chatbot modes available:\n")
print("1. 📦 Simple RAG (keyword search):")
print("   ask('What are lung cancer screening requirements?')")
print()
print("2. 🧠 Graph-Aware (relationship questions):")
print("   ask_graph('How are lung cancer screening and Medicare related?')")
print("   ask_relationship('What policies affect Medicare beneficiaries?')")
print()
print("="*70)

# Test the graph chatbot
if len(entities_df) > 0:
    print("\n🧪 Testing graph chatbot...\n")
    test_answer = ask_relationship("How are lung cancer screening and Medicare related?")

# 📖 How to Use the Chatbot

## Quick Start
Simply run: `ask('your question')`

## Example Questions

### Lung Cancer Screening
```python
ask('What are the eligibility requirements for lung cancer screening?')
ask('What age range qualifies for lung cancer screening?')
ask('How many pack-years of smoking history is required?')
```

### Telehealth
```python
ask('What are the telehealth requirements for 2026?')
ask('What services can be provided via telehealth?')
```

### Part D Medicare
```python
ask('What changed in the Part D redesign?')
ask('What are the Part D benefit changes for 2026?')
```

### Preventive Services
```python
ask('What preventive services are covered by Medicare?')
ask('Are preventive screenings covered without cost sharing?')
```

## Advanced Usage

For more control, use `ask_question()` directly:
```python
# Show source passages
answer = ask_question('your question', show_sources=True)

# Hide source passages for cleaner output  
answer = ask_question('your question', show_sources=False)
```

## What's Happening Behind the Scenes

1. **Keyword Search**: Finds the 3 most relevant text chunks from your 113 document chunks
2. **Context Building**: Combines relevant passages as context
3. **LLM Generation**: Uses Llama 3.3 70B to generate an accurate answer based on the context

## Data Source

- **Input**: 4 CMS policy documents (converted to text)
- **Processing**: Split into 113 chunks of ~1200 tokens each
- **Location**: `/Volumes/research_catalog/healthcare/policy_docs/`

---

💡 **Tip**: The chatbot only knows about the 4 CMS documents you downloaded. For questions outside these documents, it will return "No relevant information found."

In [0]:
# 💬 Ask Your Questions Here
# Simply type your question in the ask() function and run this cell!

# Example - replace with your own question:
ask('What are the lung cancer screening requirements?')

# Add more questions below:
# ask('What changed for telehealth in 2026?')
# ask('What preventive services are covered?')

In [0]:
# ask_graph('How are telehealth and ACOs related?')
ask_relationship('What policies affect Medicare beneficiaries?')

# ask_relationship('How are Part D and diabetes screening related?')
# ask_graph('What policies affect telehealth?')